## Baixando bases

In [1]:
import json
import re
from pathlib import Path


def extrair_labels_ground_truth(
    arquivos_entrada,
    arquivo_saida,
):
    labels = set()

    candidatos_label = [
        "label_high",
        "high_level_label",
        "high_label",
        "level_1_label",
        "label_level_1",
        "topic",
        "label_name",
        "label",
        "category",
    ]

    def achatar_registro(registro, prefixo=""):
        resultado = {}

        for chave, valor in registro.items():
            nome = (
                f"{prefixo}.{chave}"
                if prefixo
                else str(chave)
            )

            if isinstance(valor, dict):
                resultado.update(
                    achatar_registro(
                        valor,
                        nome,
                    )
                )
            else:
                resultado[nome] = valor

        return resultado

    def localizar_label(registro):
        mapa_colunas = {
            coluna.lower(): coluna
            for coluna in registro.keys()
        }

        for candidato in candidatos_label:
            if candidato.lower() in mapa_colunas:
                return registro[
                    mapa_colunas[candidato.lower()]
                ]

        for candidato in candidatos_label:
            sufixo = "." + candidato.lower()

            for coluna_minuscula, coluna_original in mapa_colunas.items():
                if coluna_minuscula.endswith(sufixo):
                    return registro[coluna_original]

        return None

    for arquivo_entrada in arquivos_entrada:
        caminho = Path(arquivo_entrada)

        if not caminho.exists():
            raise FileNotFoundError(
                f"Arquivo não encontrado: {caminho}"
            )

        with open(
            caminho,
            "r",
            encoding="utf-8",
        ) as arquivo:
            for linha in arquivo:
                if not linha.strip():
                    continue

                registro = json.loads(linha)
                registro = achatar_registro(registro)

                label = localizar_label(registro)

                if isinstance(label, (list, tuple)):
                    label = (
                        label[0]
                        if label
                        else None
                    )

                if label is None:
                    continue

                label = str(label).strip()

                if label.lower() in {
                    "",
                    "n/a",
                    "na",
                    "nan",
                    "none",
                    "null",
                }:
                    continue

                labels.add(label)

    labels = sorted(
        labels,
        key=str.lower,
    )

    caminho_saida = Path(arquivo_saida)

    caminho_saida.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        caminho_saida,
        "w",
        encoding="utf-8",
    ) as arquivo:
        arquivo.write(
            "\n".join(labels)
        )

    print(
        f"Arquivo criado: {caminho_saida}"
    )

    print(
        f"Quantidade de tópicos ground truth: "
        f"{len(labels)}"
    )

    return labels

In [2]:
labels_wiki = extrair_labels_ground_truth(
    arquivos_entrada=[
        "datasets/topicgpt_oficial/wiki_train.metadata.jsonl",
        "datasets/topicgpt_oficial/wiki_test.metadata.jsonl",
    ],
    arquivo_saida=(
        "resultados/"
        "w_topicos_ground_truth.txt"
    ),
)

Arquivo criado: resultados/w_topicos_ground_truth.txt
Quantidade de tópicos ground truth: 45


In [3]:
labels_bills = extrair_labels_ground_truth(
    arquivos_entrada=[
        "datasets/topicgpt_oficial/bills_train.metadata.jsonl",
        "datasets/topicgpt_oficial/bills_test.metadata.jsonl",
    ],
    arquivo_saida=(
        "resultados/"
        "b_topicos_ground_truth.txt"
    ),
)

Arquivo criado: resultados/b_topicos_ground_truth.txt
Quantidade de tópicos ground truth: 21


In [4]:
import re
from pathlib import Path


def ler_topicos_refinados(caminho_arquivo):
    caminho = Path(caminho_arquivo)

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    padrao = re.compile(
        r"^\[\d+\]\s+"
        r"(.+?)"
        r"(?:\s+\(Count:\s*\d+\))?"
        r"\s*:"
    )

    topicos = []

    with open(
        caminho,
        "r",
        encoding="utf-8",
    ) as arquivo:
        for linha in arquivo:
            conteudo = linha.strip()

            if not conteudo:
                continue

            match = padrao.match(conteudo)

            if match:
                topicos.append(
                    match.group(1).strip()
                )

    topicos = sorted(
        set(topicos),
        key=str.lower,
    )

    print(
        f"Tópicos refinados encontrados: "
        f"{len(topicos)}"
    )

    return topicos

In [5]:
topicos_wiki = ler_topicos_refinados(
    "resultados/w_top_refinados_limpo.txt"
)

topicos_bills = ler_topicos_refinados(
    "resultados/b_top_refinados_limpo.txt"
)

Tópicos refinados encontrados: 23
Tópicos refinados encontrados: 126


In [6]:
import re
import unicodedata
import pandas as pd


def normalizar_nome_topico(texto):
    texto = str(texto).lower().strip()

    texto = unicodedata.normalize(
        "NFKD",
        texto,
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(
            caractere
        )
    )

    texto = re.sub(
        r"[^a-z0-9]+",
        " ",
        texto,
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto,
    ).strip()

    return texto


def comparar_nomes_exatos(
    topicos_criados,
    topicos_ground_truth,
):
    mapa_criados = {
        normalizar_nome_topico(topico): topico
        for topico in topicos_criados
    }

    mapa_ground_truth = {
        normalizar_nome_topico(topico): topico
        for topico in topicos_ground_truth
    }

    nomes_comuns = (
        set(mapa_criados)
        .intersection(mapa_ground_truth)
    )

    resultado = pd.DataFrame(
        [
            {
                "topico_criado": mapa_criados[nome],
                "topico_ground_truth": (
                    mapa_ground_truth[nome]
                ),
                "match_exato": True,
            }
            for nome in sorted(nomes_comuns)
        ]
    )

    print(
        "Tópicos criados:",
        len(mapa_criados),
    )

    print(
        "Tópicos ground truth:",
        len(mapa_ground_truth),
    )

    print(
        "Correspondências exatas:",
        len(resultado),
    )

    return resultado

In [7]:
matches_exatos_wiki = comparar_nomes_exatos(
    topicos_criados=topicos_wiki,
    topicos_ground_truth=labels_wiki,
)

display(matches_exatos_wiki)

Tópicos criados: 23
Tópicos ground truth: 45
Correspondências exatas: 8


,topico_criado,topico_ground_truth,match_exato
0,Architecture,Architecture,True
1,Education,Education,True
2,Film,Film,True
3,Geography,Geography,True
4,Law,Law,True
5,Religion,Religion,True
6,Television,Television,True
7,Video Games,Video games,True


In [8]:
matches_exatos_bills = comparar_nomes_exatos(
    topicos_criados=topicos_bills,
    topicos_ground_truth=labels_bills,
)

display(matches_exatos_bills)


Tópicos criados: 126
Tópicos ground truth: 21
Correspondências exatas: 6


,topico_criado,topico_ground_truth,match_exato
0,Agriculture,Agriculture,True
1,Civil Rights,Civil Rights,True
2,Education,Education,True
3,Social Welfare,Social Welfare,True
4,Technology,Technology,True
5,Transportation,Transportation,True


In [9]:
import numpy as np
import pandas as pd

from sentence_transformers import (
    SentenceTransformer,
    util,
)


def comparar_topicos_semanticamente(
    topicos_criados,
    topicos_ground_truth,
    nome_dataset,
    modelo_embeddings=(
        "sentence-transformers/"
        "all-MiniLM-L6-v2"
    ),
):
    if not topicos_criados:
        raise ValueError(
            "A lista de tópicos criados está vazia."
        )

    if not topicos_ground_truth:
        raise ValueError(
            "A lista de tópicos ground truth está vazia."
        )

    modelo = SentenceTransformer(
        modelo_embeddings
    )

    embeddings_criados = modelo.encode(
        topicos_criados,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    embeddings_ground_truth = modelo.encode(
        topicos_ground_truth,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    matriz_similaridade = util.cos_sim(
        embeddings_criados,
        embeddings_ground_truth,
    ).cpu().numpy()

    resultados = []

    for indice_criado, topico_criado in enumerate(
        topicos_criados
    ):
        similaridades = (
            matriz_similaridade[indice_criado]
        )

        indices_melhores = np.argsort(
            similaridades
        )[::-1][:3]

        registro = {
            "dataset": nome_dataset,
            "topico_criado": topico_criado,
        }

        for posicao, indice_gt in enumerate(
            indices_melhores,
            start=1,
        ):
            registro[
                f"ground_truth_{posicao}"
            ] = topicos_ground_truth[indice_gt]

            registro[
                f"similaridade_{posicao}"
            ] = float(
                similaridades[indice_gt]
            )

        resultados.append(registro)

    resultado_df = pd.DataFrame(
        resultados
    )

    resultado_df = resultado_df.sort_values(
        by="similaridade_1",
        ascending=False,
    ).reset_index(drop=True)

    return resultado_df

/home/tiagosampaio/experimento_mestrado/.venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


## Comparativo

In [10]:
comparacao_wiki = (
    comparar_topicos_semanticamente(
        topicos_criados=topicos_wiki,
        topicos_ground_truth=labels_wiki,
        nome_dataset="WIKI",
    )
)

display(comparacao_wiki)


Batches: 100%|██████████| 2/2 [00:00<00:00, 65.27it/s]


,dataset,topico_criado,ground_truth_1,similaridade_1,ground_truth_2,similaridade_2,ground_truth_3,similaridade_3
0,WIKI,Geography,Geography,1.000000,Earth science,0.462875,Places,0.455529
1,WIKI,Video Games,Video games,1.000000,Television,0.481194,Football,0.476726
2,WIKI,Religion,Religion,1.000000,Philosophy,0.564598,Education,0.468916
3,WIKI,Education,Education,1.000000,Television,0.494363,Economics and business,0.471969
4,WIKI,Film,Film,1.000000,Television,0.576409,Art,0.551518
5,WIKI,Law,Law,1.000000,Politics and government,0.469360,Film,0.443360
6,WIKI,Television,Television,1.000000,Film,0.576409,Football,0.567865
7,WIKI,Architecture,Architecture,1.000000,Computing and engineering,0.535065,"Weapons, equipment, and buildings",0.456433
8,WIKI,Transportation,Transport,0.886171,Economics and business,0.446095,Education,0.431327
9,WIKI,Politics,Politics and government,0.846596,Religion,0.519760,Law,0.506543


In [11]:
comparacao_bills = (
    comparar_topicos_semanticamente(
        topicos_criados=topicos_bills,
        topicos_ground_truth=labels_bills,
        nome_dataset="BILLS",
    )
)

display(comparacao_bills)

Batches: 100%|██████████| 1/1 [00:00<00:00, 182.30it/s]


,dataset,topico_criado,ground_truth_1,similaridade_1,ground_truth_2,similaridade_2,ground_truth_3,similaridade_3
0,BILLS,Agriculture,Agriculture,1.000000,Public Lands,0.485704,Education,0.438697
1,BILLS,Civil Rights,Civil Rights,1.000000,Law and Crime,0.470575,Social Welfare,0.437393
2,BILLS,Transportation,Transportation,1.000000,Foreign Trade,0.494896,Technology,0.447882
3,BILLS,Education,Education,1.000000,Technology,0.537548,Culture,0.525146
4,BILLS,Social Welfare,Social Welfare,1.000000,Civil Rights,0.437393,Housing,0.405626
...,...,...,...,...,...,...,...,...
121,BILLS,Libraries,Transportation,0.315494,Technology,0.312653,Environment,0.293423
122,BILLS,Endangered Species,Public Lands,0.292088,Environment,0.287890,Agriculture,0.263840
123,BILLS,Drone Regulation,Government Operations,0.291025,Technology,0.260090,Public Lands,0.260032
124,BILLS,Biometrics,Technology,0.288565,Health,0.202577,Immigration,0.194691


In [12]:
from itertools import zip_longest
from pathlib import Path


def ler_lista_txt(caminho):
    caminho = Path(caminho)

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    with open(
        caminho,
        "r",
        encoding="utf-8",
    ) as arquivo:
        valores = [
            linha.strip()
            for linha in arquivo
            if linha.strip()
        ]

    return sorted(
        set(valores),
        key=str.casefold,
    )


def extrair_nome_topico_refinado(linha):
    import re

    padrao = re.compile(
        r"^\[\d+\]\s+"
        r"(.+?)"
        r"(?:\s+\(Count:\s*\d+\))?"
        r"\s*:"
    )

    match = padrao.match(linha)

    if match:
        return match.group(1).strip()

    return linha.strip()


topicos_criados_wiki_brutos = ler_lista_txt(
    "resultados/w_top_refinados_limpo.txt"
)

topicos_criados_wiki = sorted(
    {
        extrair_nome_topico_refinado(topico)
        for topico in topicos_criados_wiki_brutos
    },
    key=str.casefold,
)

topicos_ground_truth_wiki = ler_lista_txt(
    "resultados/w_topicos_ground_truth.txt"
)

largura_criados = max(
    [len("TÓPICOS CRIADOS")] +
    [len(topico) for topico in topicos_criados_wiki]
)

largura_ground_truth = max(
    [len("GROUND TRUTH")] +
    [len(topico) for topico in topicos_ground_truth_wiki]
)

separador = (
    "-" * largura_criados
    + "-+-"
    + "-" * largura_ground_truth
)

print(
    f"{'TÓPICOS CRIADOS':<{largura_criados}}"
    f" | "
    f"{'GROUND TRUTH':<{largura_ground_truth}}"
)

print(separador)

for topico_criado, topico_ground_truth in zip_longest(
    topicos_criados_wiki,
    topicos_ground_truth_wiki,
    fillvalue="",
):
    print(
        f"{topico_criado:<{largura_criados}}"
        f" | "
        f"{topico_ground_truth:<{largura_ground_truth}}"
    )

print(separador)

print(
    f"Total de tópicos criados: "
    f"{len(topicos_criados_wiki)}"
)

print(
    f"Total de tópicos ground truth: "
    f"{len(topicos_ground_truth_wiki)}"
)

TÓPICOS CRIADOS     | GROUND TRUTH                      
--------------------+-----------------------------------
Architecture        | Agriculture, food, and drink      
Biography           | Albums                            
Business            | Architecture                      
Comedy              | Armies and military units         
Education           | Art                               
Film                | Baseball and cricket              
Geography           | Basketball                        
History             | Battles, exercises, and conflicts 
Law                 | Biology and medicine              
Literature          | Chemistry and materials science   
Military            | Classical compositions            
Military History    | Computing and engineering         
Military Technology | Culture, sociology, and psychology
Music               | Earth science                     
Naval History       | Economics and business            
Politics            | Education

In [13]:
from itertools import zip_longest
from pathlib import Path


def ler_lista_txt(caminho):
    caminho = Path(caminho)

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    with open(
        caminho,
        "r",
        encoding="utf-8",
    ) as arquivo:
        valores = [
            linha.strip()
            for linha in arquivo
            if linha.strip()
        ]

    return sorted(
        set(valores),
        key=str.casefold,
    )


def extrair_nome_topico_refinado(linha):
    import re

    padrao = re.compile(
        r"^\[\d+\]\s+"
        r"(.+?)"
        r"(?:\s+\(Count:\s*\d+\))?"
        r"\s*:"
    )

    match = padrao.match(linha)

    if match:
        return match.group(1).strip()

    return linha.strip()


topicos_criados_bills_brutos = ler_lista_txt(
    "resultados/b_top_refinados_limpo.txt"
)

topicos_criados_bills = sorted(
    {
        extrair_nome_topico_refinado(topico)
        for topico in topicos_criados_bills_brutos
    },
    key=str.casefold,
)

topicos_ground_truth_bills = ler_lista_txt(
    "resultados/b_topicos_ground_truth.txt"
)

largura_criados = max(
    [len("TÓPICOS CRIADOS")] +
    [len(topico) for topico in topicos_criados_bills]
)

largura_ground_truth = max(
    [len("GROUND TRUTH")] +
    [len(topico) for topico in topicos_ground_truth_bills]
)

separador = (
    "-" * largura_criados
    + "-+-"
    + "-" * largura_ground_truth
)

print(
    f"{'TÓPICOS CRIADOS':<{largura_criados}}"
    f" | "
    f"{'GROUND TRUTH':<{largura_ground_truth}}"
)

print(separador)

for topico_criado, topico_ground_truth in zip_longest(
    topicos_criados_bills,
    topicos_ground_truth_bills,
    fillvalue="",
):
    print(
        f"{topico_criado:<{largura_criados}}"
        f" | "
        f"{topico_ground_truth:<{largura_ground_truth}}"
    )

print(separador)

print(
    f"Total de tópicos criados: "
    f"{len(topicos_criados_bills)}"
)

print(
    f"Total de tópicos ground truth: "
    f"{len(topicos_ground_truth_bills)}"
)

TÓPICOS CRIADOS              | GROUND TRUTH         
-----------------------------+----------------------
Agriculture                  | Agriculture          
Animal Welfare               | Civil Rights         
Antitrust Law                | Culture              
Aquaculture                  | Defense              
Arts & Culture               | Domestic Commerce    
Aviation                     | Education            
Aviation Safety              | Energy               
Bankruptcy Law               | Environment          
Bioethics                    | Foreign Trade        
Biometrics                   | Government Operations
Border Security              | Health               
Broadcasting                 | Housing              
Campaign Finance             | Immigration          
Charitable Giving            | International Affairs
Child Protection             | Labor                
Civil Rights                 | Law and Crime        
Climate Change               | Macroeconomics 